# Mat & Comp – Lección 11: π por Montecarlo
## Calcular π lanzando dardos al azar

**Curso:** Matemáticas y Computación · **Duración sugerida:** 2 sesiones de 50 minutos

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar el **método de Montecarlo**: usar puntos al azar para medir áreas.
2. Estimar **π** comparando el área de un círculo con la de un cuadrado.
3. Usar `numpy` para generar **millones** de números al azar en una sola línea.
4. Medir el **error** de una estimación y descubrir que crece la precisión como **1/√n**: para ganar una cifra hay que lanzar 100 veces más dardos.
5. Calcular otras áreas y la **aguja de Buffon**, un experimento del siglo XVIII para medir π con palillos.

## 🔍 ¿Se puede calcular π a ciegas?

Dibuja un cuadrado de 2 × 2 en el suelo y, dentro, un círculo de radio 1 que toque los cuatro lados. Ahora lanza granos de arroz **al azar** sobre el cuadrado, sin apuntar.

- ¿Qué fracción de los granos crees que caerá dentro del círculo?
- ¿Podrías usar esa fracción para calcular **π**, sin medir nada con regla?

✏️ *Escribe tu predicción: si lanzas 1000 granos, ¿cuántos caerán dentro del círculo?*

El método lleva el nombre del **casino de Montecarlo**, porque se basa en el azar, como la ruleta. Lo usaron en 1946 Stanisław Ulam (el de la espiral de la Lección 3) y John von Neumann para diseñar reactores nucleares.

## 📚 El concepto: áreas y probabilidades

Si un punto cae **al azar y de manera uniforme** en el cuadrado, la probabilidad de que caiga en una región es proporcional a su **área**:

$$P(\text{dentro del círculo}) = \frac{\text{área del círculo}}{\text{área del cuadrado}} = \frac{\pi \cdot 1^2}{2 \cdot 2} = \frac{\pi}{4}$$

Por la ley de los grandes números (Lección 2), la **proporción** de puntos dentro se acerca a π/4. Entonces:

$$\pi \approx 4 \cdot \frac{\text{puntos dentro}}{\text{puntos totales}}$$

¿Cómo saber si el punto (x, y) está dentro? Por **Pitágoras** (Lección 9): su distancia al centro es √(x² + y²), y está dentro si **x² + y² ≤ 1**.

## 1. Lanzar dardos uno por uno

In [ ]:
import random
import math

def estimar_pi(n):
    dentro = 0
    for _ in range(n):
        x = random.uniform(-1, 1)      # un número decimal al azar entre -1 y 1
        y = random.uniform(-1, 1)
        if x**2 + y**2 <= 1:
            dentro += 1
    return 4 * dentro / n

random.seed(11)
for n in [10, 100, 1000, 10000, 100000]:
    print(f"{n:7} dardos → π ≈ {estimar_pi(n):.5f}")
print("π real:       ", math.pi)

## 2. Ver los dardos

Ahora con `numpy`, que genera todos los puntos de una vez y es muchísimo más rápido. Mueve el slider para lanzar más dardos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def ver_dardos(n):
    rng = np.random.default_rng(2026)
    x = rng.uniform(-1, 1, n)          # n números al azar de una sola vez
    y = rng.uniform(-1, 1, n)
    dentro = x**2 + y**2 <= 1          # un arreglo de True/False
    estimado = 4 * dentro.sum() / n    # True cuenta como 1 al sumar

    fig, ax = plt.subplots(figsize=(6, 6))
    tam = 12 if n <= 500 else 2 if n <= 20000 else 0.3
    ax.scatter(x[dentro], y[dentro], s=tam, color="#e76f51")
    ax.scatter(x[~dentro], y[~dentro], s=tam, color="#457b9d")   # ~ significa "no"
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, lw=2))
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal")
    ax.set_title(f"{n} dardos: {dentro.sum()} dentro → π ≈ {estimado:.4f}")
    plt.show()
    print(f"Error: {abs(estimado - math.pi):.4f}")

interact(ver_dardos, n=IntSlider(value=100, min=10, max=100000, step=10, continuous_update=False));

**¿Qué pasa si…?**
- …lanzas solo 10 dardos? ¿Qué valores de π puedes obtener con 10 dardos? (Pista: 4 · k/10.)
- …cambias la semilla `2026` por otra? ¿Cambia mucho la estimación con 100 dardos? ¿Y con 100 000?

## 3. ¿Qué tan rápido mejora?

Calculemos el error de la estimación para cantidades de dardos cada vez mayores. Usaremos escala **logarítmica** en los dos ejes.

In [ ]:
rng = np.random.default_rng(1)
ns = np.unique(np.logspace(1, 7, 40).astype(int))      # de 10 a 10 millones
errores = []
for n in ns:
    x = rng.random(n); y = rng.random(n)                # basta con un cuarto de círculo
    est = 4 * np.mean(x**2 + y**2 <= 1)
    errores.append(abs(est - math.pi))

plt.figure(figsize=(8, 5))
plt.loglog(ns, errores, "o", label="error de una estimación")
plt.loglog(ns, 1.6 / np.sqrt(ns), "--", label="1,6 / √n")
plt.xlabel("número de dardos n"); plt.ylabel("error")
plt.title("El error de Montecarlo baja como 1/√n")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

El error disminuye aproximadamente como **1/√n**. Eso es una buena y una mala noticia:

- 👍 Funciona para casi cualquier forma, por complicada que sea.
- 👎 Es **lento**: para tener **una cifra decimal más** (error 10 veces menor) hay que lanzar **100 veces más** dardos. Con un millón de dardos solo se aciertan unas 3 cifras de π.

Por eso para calcular billones de cifras de π se usan fórmulas, no dardos. Pero para muchos problemas reales (física, finanzas, clima) Montecarlo es la mejor herramienta disponible.

In [ ]:
# Con 10 millones de dardos en numpy (tarda un par de segundos)
rng = np.random.default_rng(7)
n = 10_000_000
x = rng.random(n); y = rng.random(n)
est = 4 * np.mean(x**2 + y**2 <= 1)
print(f"π ≈ {est:.6f}   (error {abs(est - math.pi):.6f})")

## 4. Montecarlo para otras áreas

El método sirve para **cualquier** región. Por ejemplo, el área bajo la parábola y = x² entre 0 y 1: lanzamos puntos en el cuadrado [0, 1] × [0, 1] y contamos los que quedan **debajo** de la curva.

In [ ]:
def area_bajo(f, n=100000, ymax=1.0, xmin=0.0, xmax=1.0, dibujar=True):
    rng = np.random.default_rng(3)
    x = rng.uniform(xmin, xmax, n)
    y = rng.uniform(0, ymax, n)
    debajo = y <= f(x)
    area_caja = (xmax - xmin) * ymax
    area = area_caja * debajo.mean()
    if dibujar:
        m = min(n, 3000)
        plt.figure(figsize=(6, 5))
        plt.scatter(x[:m][debajo[:m]], y[:m][debajo[:m]], s=3, color="#2a9d8f")
        plt.scatter(x[:m][~debajo[:m]], y[:m][~debajo[:m]], s=3, color="lightgray")
        xs = np.linspace(xmin, xmax, 200)
        plt.plot(xs, f(xs), color="black")
        plt.title(f"Área estimada ≈ {area:.4f}")
        plt.show()
    return area

area_bajo(lambda x: x**2)
print("Valor exacto: 1/3 =", 1 / 3)

Usamos `lambda x: x**2`, una forma corta de escribir una función sin nombre.

**¿Qué pasa si…?**
- …cambias la función por `lambda x: x**3`? ¿Y por `lambda x: np.sqrt(x)`? ¿Ves un patrón con 1/3, 1/4, …?
- …calculas el área bajo `lambda x: np.sqrt(1 - x**2)`? ¿Qué figura es y cuánto debería dar?

## 5. La aguja de Buffon (1777)

El naturalista francés Georges-Louis Leclerc, conde de **Buffon**, propuso este experimento:

> Sobre un piso de tablas paralelas separadas una distancia **d**, se dejan caer agujas de longitud **ℓ ≤ d** al azar. ¿Qué probabilidad hay de que una aguja **cruce** una línea?

La respuesta, sorprendentemente, contiene a π:

$$P(\text{cruza}) = \frac{2\ell}{\pi d} \qquad\Rightarrow\qquad \pi \approx \frac{2 \ell \cdot \text{agujas lanzadas}}{d \cdot \text{agujas que cruzan}}$$

¡Puedes medir π en clase con palillos y un piso de baldosas!

In [ ]:
def buffon(n, l=1.0, d=1.0, semilla=0):
    rng = np.random.default_rng(semilla)
    centro_y = rng.uniform(0, 10 * d, n)              # altura del centro de la aguja
    centro_x = rng.uniform(0, 10, n)
    angulo = rng.uniform(0, np.pi, n)                  # inclinación de la aguja
    y1 = centro_y - (l / 2) * np.sin(angulo)
    y2 = centro_y + (l / 2) * np.sin(angulo)
    cruza = np.floor(y1 / d) != np.floor(y2 / d)       # los extremos están en tablas distintas
    return centro_x, centro_y, angulo, cruza

def ver_buffon(n):
    l, d = 1.0, 1.0
    cx, cy, ang, cruza = buffon(n, l, d)
    fig, ax = plt.subplots(figsize=(7, 7))
    for k in range(11):
        ax.axhline(k * d, color="#bc6c25", lw=1)
    for i in range(min(n, 400)):
        dx, dy = (l / 2) * np.cos(ang[i]), (l / 2) * np.sin(ang[i])
        ax.plot([cx[i] - dx, cx[i] + dx], [cy[i] - dy, cy[i] + dy],
                color="crimson" if cruza[i] else "#264653", lw=1.2)
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect("equal"); ax.axis("off")
    estimado = 2 * l * n / (d * cruza.sum()) if cruza.sum() > 0 else float("nan")
    ax.set_title(f"{n} agujas, {cruza.sum()} cruzan (rojas) → π ≈ {estimado:.4f}")
    plt.show()

interact(ver_buffon, n=IntSlider(value=100, min=10, max=100000, step=10, continuous_update=False));

(Si lanzas más de 400 agujas, solo se dibujan las primeras 400, pero se cuentan todas.)

**¿Qué pasa si…?**
- …usas agujas más cortas, por ejemplo ℓ = 0,5? ¿Cuántas cruzan? ¿Sigue funcionando la fórmula?

---
## ✍️ Ejercicios

Resuelve cada ejercicio en la celda de código que está debajo. Puedes reutilizar `estimar_pi`, `area_bajo` y `buffon`, y `np` (si reiniciaste Colab, vuelve a ejecutar las celdas anteriores).

### Ejercicio 1 (básico) · ¿Cuánto varía?
Repite 10 veces `estimar_pi(100)` y anota los resultados. Luego repite 10 veces `estimar_pi(10000)`.
a) ¿Cuál es el valor más pequeño y el más grande en cada caso?
b) ¿En cuál de los dos casos los resultados están más "apretados"?

In [ ]:
# Escribe tu código aquí

### Ejercicio 2 (básico) · El área de la curva seno
Usa `area_bajo` para estimar el área bajo y = sen(x) entre 0 y π (`np.sin`, con `xmin=0`, `xmax=np.pi`, `ymax=1`).
¿A qué número entero parece acercarse?

In [ ]:
# Escribe tu código aquí

### Ejercicio 3 (intermedio) · Dardos contra cuadrícula
En vez de puntos al azar, usa una **cuadrícula** regular de k × k puntos en [0, 1] × [0, 1] (con `np.linspace` y `np.meshgrid`) y cuenta cuántos caen dentro del cuarto de círculo.
a) Estima π con k = 10, 100 y 1000.
b) Compara con Montecarlo usando la **misma cantidad** de puntos (k²). ¿Cuál es más preciso en 2D?

In [ ]:
# Escribe tu código aquí

### Ejercicio 4 (intermedio) · π en tres dimensiones
Una esfera de radio 1 cabe justa dentro de un cubo de lado 2. La razón entre sus volúmenes es (4/3)π / 8 = **π/6**.
a) Lanza puntos (x, y, z) al azar en el cubo y estima π contando los que cumplen x² + y² + z² ≤ 1.
b) ¿Es más o menos preciso que en 2D con la misma cantidad de puntos?

In [ ]:
# Escribe tu código aquí

### Ejercicio 5 (avanzado) · La campana de las estimaciones
Haz 2000 estimaciones de π, cada una con n = 1000 dardos (con numpy es rápido).
a) Dibuja un **histograma** de las 2000 estimaciones con `plt.hist`. ¿Qué forma tiene?
b) Calcula la **desviación estándar** con `np.std`. Repite con n = 4000. ¿Por cuánto se dividió? ¿Coincide con la regla 1/√n?

In [ ]:
# Escribe tu código aquí

---
## 🏆 Reto final (opcional) · Buffon en el salón y el caso Lazzarini

1. **Experimento real:** marquen líneas paralelas en una cartulina separadas por la longitud de un palillo. Cada estudiante lanza 20 palillos y cuenta cuántos cruzan. Sumen los resultados del salón y estimen π.
2. En 1901, el matemático italiano **Mario Lazzarini** dijo haber lanzado 3408 agujas y obtenido **π ≈ 355/113 = 3,1415929…**, ¡con 6 cifras correctas!
   - Usa la regla del error 1/√n: ¿cuántas cifras correctas esperarías con 3408 agujas?
   - Con ℓ/d = 5/6 y 3408 agujas, ¿cuántas tienen que cruzar para obtener exactamente 355/113?
   - Simula 1000 experimentos de 3408 agujas. ¿En cuántos sale una estimación tan buena? ¿Crees que Lazzarini hizo trampa?

In [ ]:
# Escribe tu código aquí

---
## 🎯 Cierre: lo que aprendiste hoy

- El **método de Montecarlo** mide áreas con puntos al azar: la proporción de puntos dentro de una región se acerca a su área relativa.
- Con un círculo dentro de un cuadrado, **π ≈ 4 · dentro / total**; para saber si un punto está dentro se usa Pitágoras: x² + y² ≤ 1.
- El error disminuye como **1/√n**: cada cifra extra cuesta 100 veces más puntos.
- Montecarlo sirve para cualquier forma: áreas bajo curvas, volúmenes y la **aguja de Buffon**.
- En Python aprendiste: `random.uniform`, `np.random.default_rng`, arreglos de True/False como filtros (`x[dentro]`, `~`), `np.mean`, `lambda`, `plt.loglog` y `plt.hist`.

**Para pensar:** si alguien te dice "con mi computadora calculé π lanzando mil millones de dardos", ¿cuántas cifras correctas esperarías que tenga su resultado? ¿Es un buen método para calcular π?